# DB 검색 조회 속도 측정 (Colab)

로컬 PC 대신 Colab VM에 PostgreSQL 16 + PostGIS + pgvector를 깔고, 데이터 버전 1 덤프를 복원해 검색 조회(`docs/db/DB_TABLES.md` 10-1절 ①②③⑤)의 속도를 잽니다. 설명: `docs/db/DB_SEARCH_BENCH.md`.

**Drive에 미리 둘 것** (`내 드라이브/store-search-ai/`)

| 경로 | 내용 |
|---|---|
| `service_load/storesearch_v1.dump` | 데이터 버전 1 덤프(`docs/dev/DATA_LOAD.md` 4절, 약 1.2GB) |
| `service_load/bench/service_queries_v1.csv` | 평가 질의 695개(검색어 + 검색 위치). 로컬 `data/bench/` 또는 AI 레포 `data/query/` |
| `runs/finetune/bge_m3_ft_mj_20261014_0743/` | 서비스 모델(이미 있음). 질의 벡터를 만듦 |

**결과**: `service_load/bench/db_search_<시각>.md` (표). 이 파일을 PR·문서에 옮깁니다.

- DB 설정은 로컬 `infra/docker-compose.yml`과 같게 둡니다(shared_buffers 2GB, work_mem 64MB).
- 클라이언트(이 노트북)와 DB가 같은 VM이라, 시간은 "API 서버와 DB가 한 서버"일 때와 비슷합니다. Colab CPU 수(보통 2개)가 실제 서버보다 적을 수 있어 **보수적인 값**입니다.
- 런타임: GPU(T4). GPU는 질의 벡터 만들 때만 씁니다. 전체 약 15~20분(설치 3분, 덤프 복사·복원 5분, 측정 5~10분).

In [ ]:
import os, subprocess, multiprocessing
print("CPU", multiprocessing.cpu_count(), "개")
print(subprocess.run("free -g | head -2; lsb_release -ds; nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%bash
# PostgreSQL 16 + PostGIS 3 + pgvector (PGDG 저장소)
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y curl ca-certificates lsb-release > /dev/null
install -d /usr/share/postgresql-common/pgdg
curl -s -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get -qq update > /dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql-16 postgresql-16-postgis-3 postgresql-16-pgvector > /dev/null
pg_ctlcluster 16 main start || true
runuser -u postgres -- psql -qc "ALTER SYSTEM SET shared_buffers = '2GB'" -c "ALTER SYSTEM SET work_mem = '64MB'" \
  -c "ALTER SYSTEM SET maintenance_work_mem = '1GB'" -c "ALTER USER postgres PASSWORD 'bench'"
pg_ctlcluster 16 main restart
runuser -u postgres -- psql -tAc "select version()"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/store-search-ai")
DUMP = ROOT / "service_load/storesearch_v1.dump"
QUERIES = ROOT / "service_load/bench/service_queries_v1.csv"
MODEL_DIR = ROOT / "runs/finetune/bge_m3_ft_mj_20261014_0743"
OUT_DIR = ROOT / "service_load/bench"
for p in [DUMP, QUERIES, MODEL_DIR]:
    assert p.exists(), f"Drive에 없음: {p}"
print(f"덤프 {DUMP.stat().st_size / 1e9:.2f}GB")

In [ ]:
%%bash
# Drive에서 로컬 디스크로 복사한 뒤 복원(Drive에서 바로 읽으면 느림). 덤프에 Flyway 기록·확장까지 들어 있음
set -e
cp "/content/drive/MyDrive/store-search-ai/service_load/storesearch_v1.dump" /tmp/storesearch_v1.dump
chmod 644 /tmp/storesearch_v1.dump
runuser -u postgres -- dropdb --if-exists storesearch
runuser -u postgres -- createdb storesearch
runuser -u postgres -- pg_restore -d storesearch --no-owner -j 2 /tmp/storesearch_v1.dump 2> /tmp/restore.err || true
echo "복원 경고·오류 $(grep -c ERROR /tmp/restore.err || true)건(public 스키마 등 이미 있는 것은 무시해도 됨)"; grep ERROR /tmp/restore.err | head -5 || true
runuser -u postgres -- psql -d storesearch -tAc "select version_name, status, n_total from data_version; select count(*) from store_snapshot"

In [ ]:
!pip install -q "psycopg[binary,pool]" sentence-transformers==5.7.0 transformers==5.18.0

In [ ]:
# 질의 벡터: 임베딩 서버와 같은 설정(fp32, 128토큰, 정규화)
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

qs = pd.read_csv(QUERIES, dtype=str, keep_default_na=False, encoding="utf-8-sig")
qs = qs[(qs["status"] == "active") & (qs["latitude"] != "") & (qs["longitude"] != "")].reset_index(drop=True)
qs["lat"] = qs["latitude"].astype(float)
qs["lng"] = qs["longitude"].astype(float)
model = SentenceTransformer(str(MODEL_DIR), device="cuda", model_kwargs={"dtype": torch.float32})
model.max_seq_length = 128
qv = model.encode(qs["query"].tolist(), batch_size=128, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
qs["vec"] = ["[" + ",".join(f"{x:.8g}" for x in v) + "]" for v in qv]
del model
torch.cuda.empty_cache()
print(len(qs), "개 질의", qs[["query", "lat", "lng"]].head(3).to_string())

In [ ]:
# DB를 다시 시작해 공유 메모리를 비운 뒤 잰다(1회차 = 메모리에 아직 없는 상태에 가까움)
import subprocess, time
subprocess.run("pg_ctlcluster 16 main restart", shell=True, check=True)
time.sleep(2)

import psycopg
DSN = "host=127.0.0.1 dbname=storesearch user=postgres password=bench"
conn = psycopg.connect(DSN, autocommit=True)
V = conn.execute("select version_id from data_version where status = 'active'").fetchone()[0]
settings = {k: conn.execute(f"show {k}").fetchone()[0] for k in ["shared_buffers", "work_mem", "max_parallel_workers_per_gather"]}
pg = conn.execute("select extversion from pg_extension where extname = 'vector'").fetchone()[0]
pgis = conn.execute("select postgis_lib_version()").fetchone()[0]
print("버전", V, settings, "pgvector", pg, "postgis", pgis)

# 질의 위치에서 가장 가까운 가게의 법정동 코드(② 시군구·시도 범위용)
qs["code"] = [conn.execute(
    "select legal_dong_code from store_snapshot where version_id = %s and searchable "
    "order by geom <-> ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography limit 1", (V, r.lng, r.lat)).fetchone()[0]
    for r in qs.itertuples()]

In [ ]:
# DB_TABLES.md 10-1절과 같은 쿼리
TOP50 = " order by s.embedding <=> %(v)s::vector limit 50"
SEL = "select s.store_id, 1 - (s.embedding <=> %(v)s::vector) as cosine from store_snapshot s where s.version_id = %(ver)s and s.searchable "
RADIUS = "and ST_DWithin(s.geom, ST_SetSRID(ST_MakePoint(%(lng)s, %(lat)s), 4326)::geography, %(r)s)"
PREFIX = "and s.legal_dong_code like %(p)s || '%%'"
BOX = "and s.geom && ST_MakeEnvelope(%(w)s, %(s)s, %(e)s, %(n)s, 4326)::geography"
NEAREST = ("select s.store_id, s.store_name, s.primary_category, s.market_type, s.market_name, s.payment_codes, "
           "ST_Y(s.geom::geometry) as lat, ST_X(s.geom::geometry) as lng from store_snapshot s "
           "where s.version_id = %(ver)s and s.searchable " + BOX +
           " order by s.geom <-> ST_SetSRID(ST_MakePoint(%(lng)s, %(lat)s), 4326)::geography, s.store_name collate \"C\", s.store_id limit 300")
COUNT = "select count(*) from store_snapshot s where s.version_id = %(ver)s and s.searchable "
CITY, WIDE = (0.03, 0.02), (0.15, 0.10)   # 지도 화면 반폭(경도, 위도): 약 5.4×4.4km, 27×22km


def box(r, size):
    return {"w": r.lng - size[0], "s": r.lat - size[1], "e": r.lng + size[0], "n": r.lat + size[1]}


CASES = [
    ("① 반경 1km", "검색 중심 원 안 상위 50", SEL + RADIUS + TOP50, lambda r: {"r": 1000}, COUNT + RADIUS),
    ("① 반경 10km", "확장 단계", SEL + RADIUS + TOP50, lambda r: {"r": 10000}, COUNT + RADIUS),
    ("② 시군구", "법정동 코드 앞 5자리 전체", SEL + PREFIX + TOP50, lambda r: {"p": r.code[:5]}, COUNT + PREFIX),
    ("② 시도", "법정동 코드 앞 2자리 전체", SEL + PREFIX + TOP50, lambda r: {"p": r.code[:2]}, COUNT + PREFIX),
    ("③ 지도 도시 화면", "화면 안 상위 50", SEL + BOX + TOP50, lambda r: box(r, CITY), COUNT + BOX),
    ("③ 지도 넓은 화면", "화면 안 상위 50", SEL + BOX + TOP50, lambda r: box(r, WIDE), COUNT + BOX),
    ("⑤ 검색어 없는 도시 화면", "가까운 순 300", NEAREST, lambda r: box(r, CITY), None),
    ("⑤ 검색어 없는 넓은 화면", "가까운 순 300", NEAREST, lambda r: box(r, WIDE), None),
]


def params(r, extra):
    return {"v": r.vec, "ver": V, "lng": r.lng, "lat": r.lat, **extra(r)}


def timed(c, sql, p):
    t = time.perf_counter()
    c.execute(sql, p).fetchall()
    return (time.perf_counter() - t) * 1000


def pct(a, q):
    return float(np.percentile(a, q, method="higher"))


def fmt(x):
    return f"{x:.1f}"

In [ ]:
report = []
report.append(f"# DB 검색 조회 속도 — Colab ({time.strftime('%Y-%m-%d %H:%M')} UTC)\n")
report.append(f"- 환경: Colab CPU {multiprocessing.cpu_count()}개, PostgreSQL 16, pgvector {pg}, PostGIS {pgis}, "
              f"shared_buffers {settings['shared_buffers']}, work_mem {settings['work_mem']}, "
              f"max_parallel_workers_per_gather {settings['max_parallel_workers_per_gather']}")
report.append(f"- 데이터 버전 {V}(검색 대상 224,677곳), 질의 {len(qs)}개(평가 질의의 검색어 + 검색 위치)")
report.append("- 시간 = 쿼리를 보내고 결과 행을 다 받을 때까지(ms). 클라이언트와 DB가 같은 VM\n")

report.append("## 범위 안 가게 수\n\n| 경우 | 중앙값 | p90 | 최대 |\n|---|---|---|---|")
for name, _, _, extra, count_sql in CASES:
    if count_sql:
        n = [conn.execute(count_sql, params(r, extra)).fetchone()[0] for r in qs.itertuples()]
        report.append(f"| {name} | {int(np.median(n))} | {int(pct(n, 90))} | {max(n)} |")

for label in ["1회차(DB 재시작 직후)", "2회차(데이터가 메모리에 올라온 뒤)"]:
    report.append(f"\n## {label}\n\n| 경우 | 설명 | p50 | p95 | p99 | 최대 |\n|---|---|---|---|---|---|")
    for name, desc, sql, extra, _ in CASES:
        ms = [timed(conn, sql, params(r, extra)) for r in qs.itertuples()]
        report.append(f"| {name} | {desc} | {fmt(pct(ms, 50))} | {fmt(pct(ms, 95))} | {fmt(pct(ms, 99))} | {fmt(max(ms))} |")
    print("\n".join(report[-len(CASES) - 3:]))

In [ ]:
# 가장 무거운 경우: 가게가 가장 많은 시도 전체, 서울에서 가장 빽빽한 곳의 10km (같은 쿼리 30번, 첫 번 제외)
v = qs["vec"].iloc[0]
sido, sido_n = conn.execute("select left(legal_dong_code, 2), count(*) from store_snapshot where version_id = %s and searchable "
                            "group by 1 order by 2 desc limit 1", (V,)).fetchone()
lng, lat, dense_n = conn.execute(
    "select ST_X(geom::geometry), ST_Y(geom::geometry), (select count(*) from store_snapshot t where t.version_id = s.version_id "
    "and t.searchable and ST_DWithin(t.geom, s.geom, 10000)) as n from store_snapshot s tablesample system (1) "
    "where version_id = %s and searchable and legal_dong_code like '11%%' order by n desc limit 1", (V,)).fetchone()
heavy = [(f"시도 전체(코드 {sido})", sido_n, SEL + PREFIX + TOP50, {"v": v, "ver": V, "p": sido}),
         ("서울 가장 빽빽한 곳 10km", dense_n, SEL + RADIUS + TOP50, {"v": v, "ver": V, "lng": lng, "lat": lat, "r": 10000})]
report.append(f"\n## 가장 무거운 경우(질의 '{qs['query'].iloc[0]}', 30번)\n\n| 경우 | 범위 안 가게 | p50 | p95 | 최대 |\n|---|---|---|---|---|")
for name, n, sql, p in heavy:
    timed(conn, sql, p)
    ms = [timed(conn, sql, p) for _ in range(30)]
    report.append(f"| {name} | {n} | {fmt(pct(ms, 50))} | {fmt(pct(ms, 95))} | {fmt(max(ms))} |")
plan = conn.execute("explain (analyze, buffers, costs off) " + heavy[0][2], heavy[0][3]).fetchall()
report.append("\n```\n" + "\n".join(row[0] for row in plan) + "\n```")
print("\n".join(report[-6:]))

In [ ]:
# 동시 부하: 초당 RATE건을 SECONDS초 동안 일정 간격으로(앞 요청을 기다리지 않음). 검색 1건 = 1km + 10km 두 쿼리(확장까지 가는 무거운 쪽)
from concurrent.futures import ThreadPoolExecutor
from psycopg_pool import ConnectionPool

RATE, SECONDS = 20, 60   # NFR-PRF-03: 초당 20건
pool = ConnectionPool(DSN, min_size=10, max_size=10, kwargs={"autocommit": True})
pool.wait()
rows = list(qs.itertuples())
lat_ms, errors = [], []


def one(i):
    r = rows[i % len(rows)]
    t = time.perf_counter()
    try:
        with pool.connection() as c:
            c.execute(CASES[0][2], params(r, CASES[0][3])).fetchall()
            c.execute(CASES[1][2], params(r, CASES[1][3])).fetchall()
        lat_ms.append((time.perf_counter() - t) * 1000)
    except Exception as e:
        errors.append(repr(e))


with ThreadPoolExecutor(32) as ex:
    start = time.perf_counter()
    for i in range(RATE * SECONDS):
        delay = start + i / RATE - time.perf_counter()
        if delay > 0:
            time.sleep(delay)
        ex.submit(one, i)
pool.close()
report.append(f"\n## 동시 부하: 초당 {RATE}건 × {SECONDS}초(검색 1건 = 1km + 10km)\n\n| 완료 | 오류 | p50 | p95 | p99 | 최대 |\n|---|---|---|---|---|---|")
report.append(f"| {len(lat_ms)} | {len(errors)} | {fmt(pct(lat_ms, 50))} | {fmt(pct(lat_ms, 95))} | {fmt(pct(lat_ms, 99))} | {fmt(max(lat_ms))} |")
print("\n".join(report[-3:]), errors[:3])

In [ ]:
out = OUT_DIR / f"db_search_{time.strftime('%Y%m%d_%H%M')}.md"
out.write_text("\n".join(report) + "\n", encoding="utf-8")
print(out)
print("\n".join(report))